# Benchmarking convergence for TMDD model

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

%load_ext autoreload
%autoreload 2

from vpop_calibration import *


In [ ]:
# Make sure to run  `generate_synthetic_data.R` first
train_df_2d = pd.read_csv("./data/gp_training.csv")
train_df_1d = train_df_2d.loc[train_df_2d["protocol_arm"] == "arm_1"]
# Make sure this is aligned with the available observed data
df = pd.read_csv(f"./data/obs_data_100.csv")
df.drop(df.loc[df.evid == 1].index, inplace=True)
obs_df_2d = df[["id", "protocol_arm", "time", "DV"]]
obs_df_2d = obs_df_2d.loc[~obs_df_2d["DV"].isna()]
obs_df_2d = obs_df_2d.rename(columns={"DV": "value"})
obs_df_2d["output_name"] = "DV"
patients_df_2d = obs_df_2d[["id", "protocol_arm"]].drop_duplicates()

obs_df_1d = obs_df_2d.loc[obs_df_2d["protocol_arm"] == "arm_1"]
patients_df_1d = obs_df_1d[["id", "protocol_arm"]].drop_duplicates()

In [ ]:
def train_model(input_df: pd.DataFrame) -> GP:
    descriptors = ["R0", "k_eL", "Vc"]
    gp = GP(
        input_df,
        descriptors=descriptors + ["time"],
        deep_kernel=False,
        nb_inducing_points=100,
        min_delta=0.01,
        nb_training_iter=500,
        log_inputs=descriptors,
        log_outputs=[],
        learning_rate=0.1,
        lr_decay=0.995,
        plot_frame_rate=10,
    )
    gp.train()

    return gp

In [ ]:
gp_2d = train_model(train_df_2d)
gp_1d = train_model(train_df_1d)

In [ ]:
def run_saem(gp, obs, patients):
    structural_gp = StructuralGp(gp)
    # NLME model parameters
    init_log_MI = {}
    init_log_PDU = {
        "k_eL": {"mean": 0.0, "sd": 0.5},
        "R0": {"mean": 0.0, "sd": 0.5},
        "Vc": {"mean": 0.0, "sd": 0.5},
    }

    error_model_type = "additive"
    init_res_var = [0.5]

    nlme_surrogate = NlmeModel(
        structural_gp,
        patients,
        init_log_MI,
        init_log_PDU,
        init_res_var,
        None,
        error_model_type,
        num_chains=5,
        # constraints=structural_gp.training_ranges,
    )

    optimizer = PySaem(
        nlme_surrogate,
        obs,
        nb_phase1_iterations=100,
        nb_phase2_iterations=100,
        mcmc_nb_transitions=5,
        mcmc_first_burn_in=10,
        verbose=False,
        live_plot=True,
    )
    optimizer.run()
    nlme_surrogate.compute_ebe(samples_cond_dist=100, override=True)
    map_df = nlme_surrogate.map_estimates_descriptors()
    pop = np.concat(
        [
            nlme_surrogate.population_betas.numpy(),
            nlme_surrogate.omega_pop.diag().numpy(),
        ]
    )
    return pop, map_df

In [ ]:
pop_2d, ebe_2d = run_saem(gp_2d, obs_df_2d, patients_df_2d)

In [ ]:
pop_1d, ebe_1d = run_saem(gp_1d, obs_df_1d, patients_df_1d)

In [ ]:
ebe_1d.to_csv("outputs/ebe_pysaem_1dose.csv", index=False)
ebe_2d.to_csv("outputs/ebe_pysaem_2dose.csv", index=False)